# Unified Inference Notebook
Melakukan klasifikasi lengkap menggunakan semua model yang tersedia:
- Fruit Classification (PyTorch MobileNet)
- Rotten vs Fresh Classification (Keras/TensorFlow)
- Fruit Dimension Measurement (OpenCV)

In [1]:
# Import Libraries
import os
import torch
import torch.nn as nn
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt
from torchvision import transforms, models
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image as tf_image
import time
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Set Device dan Paths
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

FRUIT_CLASS_MODEL_PATH = "fruit_classification/Fruit_Classification.pth"
ROTTEN_FRESH_MODEL_PATH = "rottenvsfresh_classifier/rottenvsfresh_model.h5"

# Fruit classes untuk classification model
FRUIT_CLASSES = ['apple', 'banana', 'orange']

# Classes untuk rotten vs fresh model
ROTTEN_FRESH_CLASSES = [
    'apple_fresh',
    'apple_rotten',
    'banana_fresh',
    'banana_rotten',
    'orange_fresh',
    'orange_rotten'
]

Device: cpu


In [3]:
# ============= FRUIT CLASSIFICATION MODEL =============
class FruitClassifier:
    def __init__(self, model_path, device):
        self.device = device
        self.model = models.mobilenet_v2(pretrained=False)
        self.model.classifier[1] = nn.Linear(self.model.classifier[1].in_features, len(FRUIT_CLASSES))
        
        # Load weights
        checkpoint = torch.load(model_path, map_location=device)
        if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
            self.model.load_state_dict(checkpoint['model_state_dict'])
        else:
            self.model.load_state_dict(checkpoint)
        
        self.model.to(device)
        self.model.eval()
        
        # Transformations (sesuai dengan training)
        self.transform = transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
    
    def predict(self, image_path):
        img = Image.open(image_path).convert('RGB')
        img_tensor = self.transform(img).unsqueeze(0).to(self.device)
        
        with torch.no_grad():
            outputs = self.model(img_tensor)
            probs = nn.functional.softmax(outputs, dim=1)
            pred_class = torch.argmax(probs, dim=1).item()
            confidence = probs[0][pred_class].item()
        
        return {
            'class': FRUIT_CLASSES[pred_class],
            'confidence': confidence,
            'probabilities': {cls: prob.item() for cls, prob in zip(FRUIT_CLASSES, probs[0])}
        }

# Load model
print("Loading Fruit Classification Model...")
try:
    fruit_classifier = FruitClassifier(FRUIT_CLASS_MODEL_PATH, DEVICE)
    print("✓ Fruit Classification Model loaded successfully")
except Exception as e:
    print(f"✗ Error loading Fruit Classification Model: {e}")
    fruit_classifier = None

Loading Fruit Classification Model...
✗ Error loading Fruit Classification Model: Error(s) in loading state_dict for MobileNetV2:
	Missing key(s) in state_dict: "classifier.1.weight", "classifier.1.bias". 
	Unexpected key(s) in state_dict: "classifier.1.1.weight", "classifier.1.1.bias". 


In [4]:
# ============= ROTTEN VS FRESH MODEL =============
class RottenFreshClassifier:
    def __init__(self, model_path):
        self.model = load_model(model_path)
        
        # Transformations
        self.target_size = (224, 224)
    
    def predict(self, image_path):
        img = tf_image.load_img(image_path, target_size=self.target_size)
        img_array = tf_image.img_to_array(img)
        img_array = img_array / 255.0  # Normalize
        img_array = np.expand_dims(img_array, axis=0)
        
        predictions = self.model.predict(img_array, verbose=0)
        pred_class = np.argmax(predictions[0])
        confidence = predictions[0][pred_class]
        
        return {
            'class': ROTTEN_FRESH_CLASSES[pred_class],
            'confidence': float(confidence),
            'probabilities': {cls: float(prob) for cls, prob in zip(ROTTEN_FRESH_CLASSES, predictions[0])}
        }

# Load model
print("Loading Rotten vs Fresh Model...")
try:
    rotten_fresh_classifier = RottenFreshClassifier(ROTTEN_FRESH_MODEL_PATH)
    print("✓ Rotten vs Fresh Model loaded successfully")
except Exception as e:
    print(f"✗ Error loading Rotten vs Fresh Model: {e}")
    rotten_fresh_classifier = None

Loading Rotten vs Fresh Model...


✓ Rotten vs Fresh Model loaded successfully


In [5]:
# ============= FRUIT DIMENSION MEASUREMENT =============
class FruitDimensionMeasurer:
    def __init__(self):
        pass
    
    def preprocess(self, image):
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        blur = cv2.GaussianBlur(gray, (5, 5), 0)
        return blur
    
    def segment(self, blur):
        _, thresh = cv2.threshold(blur, 120, 255, cv2.THRESH_BINARY_INV)
        return thresh
    
    def get_largest_contour(self, thresh):
        contours, _ = cv2.findContours(
            thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )
        
        if len(contours) == 0:
            return None
        
        contours = [c for c in contours if cv2.contourArea(c) > 1000]
        
        if len(contours) == 0:
            return None
        
        return max(contours, key=cv2.contourArea)
    
    def measure_rotated(self, contour):
        rect = cv2.minAreaRect(contour)
        (cx, cy), (w, h), angle = rect
        
        if h > w:
            w, h = h, w
        
        return w, h, rect
    
    def measure(self, image_path, visualize=False):
        image = cv2.imread(image_path)
        if image is None:
            return None
        
        blur = self.preprocess(image)
        thresh = self.segment(blur)
        contour = self.get_largest_contour(thresh)
        
        if contour is None:
            return None
        
        w, h, rect = self.measure_rotated(contour)
        
        result = {
            "width_pixels": float(w),
            "height_pixels": float(h)
        }
        
        if visualize:
            vis = image.copy()
            box = cv2.boxPoints(rect)
            box = np.int32(box)
            cv2.drawContours(vis, [box], 0, (0, 255, 0), 2)
            
            text = f"{w:.1f} x {h:.1f}"
            cv2.putText(vis, text, (20, 40),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        1, (0, 255, 0), 2)
            
            plt.figure(figsize=(12, 4))
            plt.subplot(1, 3, 1)
            plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
            plt.title("Original Image")
            plt.axis('off')
            
            plt.subplot(1, 3, 2)
            plt.imshow(thresh, cmap='gray')
            plt.title("Thresholded Image")
            plt.axis('off')
            
            plt.subplot(1, 3, 3)
            plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
            plt.title("Measurement Result")
            plt.axis('off')
            plt.tight_layout()
            plt.show()
        
        return result

# Initialize
print("Initializing Fruit Dimension Measurer...")
dimension_measurer = FruitDimensionMeasurer()
print("✓ Fruit Dimension Measurer ready")

Initializing Fruit Dimension Measurer...
✓ Fruit Dimension Measurer ready


In [6]:
# ============= UNIFIED INFERENCE FUNCTION =============
def perform_inference(image_path, measure_dimensions=True, visualize_dims=False):
    """
    Melakukan inference lengkap pada image menggunakan semua model
    
    Parameters:
    -----------
    image_path : str
        Path ke image yang akan di-classify
    measure_dimensions : bool
        Apakah akan mengukur dimensi buah (memerlukan preprocessing khusus)
    visualize_dims : bool
        Apakah akan menampilkan visualisasi dimensi
    
    Returns:
    --------
    dict : Hasil klasifikasi dari semua model
    """
    
    if not os.path.exists(image_path):
        print(f"✗ Image not found: {image_path}")
        return None
    
    print(f"\n{'='*60}")
    print(f"Inferencing: {image_path}")
    print(f"{'='*60}")
    
    results = {
        'image_path': image_path,
        'timestamp': time.strftime("%Y-%m-%d %H:%M:%S")
    }
    
    # Display Image
    try:
        img = Image.open(image_path)
        plt.figure(figsize=(4, 4))
        plt.imshow(img)
        plt.title("Input Image")
        plt.axis('off')
        plt.show()
    except:
        pass
    
    # 1. Fruit Classification
    if fruit_classifier is not None:
        print("\n[1] Fruit Classification (PyTorch MobileNet)")
        print("-" * 60)
        try:
            start_time = time.time()
            fruit_pred = fruit_classifier.predict(image_path)
            inference_time = time.time() - start_time
            
            print(f"Predicted Class: {fruit_pred['class']}")
            print(f"Confidence: {fruit_pred['confidence']:.4f}")
            print(f"Inference Time: {inference_time:.4f}s")
            print("\nAll Probabilities:")
            for cls, prob in fruit_pred['probabilities'].items():
                print(f"  {cls}: {prob:.4f}")
            
            results['fruit_classification'] = fruit_pred
            results['fruit_classification']['inference_time'] = inference_time
        except Exception as e:
            print(f"✗ Error: {e}")
            results['fruit_classification'] = None
    
    # 2. Rotten vs Fresh Classification
    if rotten_fresh_classifier is not None:
        print("\n[2] Rotten vs Fresh Classification (Keras)")
        print("-" * 60)
        try:
            start_time = time.time()
            rotten_fresh_pred = rotten_fresh_classifier.predict(image_path)
            inference_time = time.time() - start_time
            
            print(f"Predicted Class: {rotten_fresh_pred['class']}")
            print(f"Confidence: {rotten_fresh_pred['confidence']:.4f}")
            print(f"Inference Time: {inference_time:.4f}s")
            print("\nAll Probabilities:")
            for cls, prob in rotten_fresh_pred['probabilities'].items():
                print(f"  {cls}: {prob:.4f}")
            
            results['rotten_fresh_classification'] = rotten_fresh_pred
            results['rotten_fresh_classification']['inference_time'] = inference_time
        except Exception as e:
            print(f"✗ Error: {e}")
            results['rotten_fresh_classification'] = None
    
    # 3. Fruit Dimension Measurement
    if measure_dimensions:
        print("\n[3] Fruit Dimension Measurement (OpenCV)")
        print("-" * 60)
        try:
            start_time = time.time()
            dimensions = dimension_measurer.measure(image_path, visualize=visualize_dims)
            inference_time = time.time() - start_time
            
            if dimensions:
                print(f"Width (pixels): {dimensions['width_pixels']:.2f}")
                print(f"Height (pixels): {dimensions['height_pixels']:.2f}")
                print(f"Inference Time: {inference_time:.4f}s")
                results['dimensions'] = dimensions
                results['dimensions']['inference_time'] = inference_time
            else:
                print("✗ Could not detect fruit in image")
                results['dimensions'] = None
        except Exception as e:
            print(f"✗ Error: {e}")
            results['dimensions'] = None
    
    print(f"\n{'='*60}")
    return results

## Contoh Penggunaan

Untuk melakukan inference pada sebuah image, gunakan fungsi `perform_inference()` dengan path ke image:

```python
# Hasil inference untuk 1 image
results = perform_inference('path/to/image.jpg', measure_dimensions=True, visualize_dims=False)
```

Modifikasi parameter:
- `measure_dimensions=True`: Mengukur dimensi buah (opsional)
- `visualize_dims=True`: Menampilkan visualisasi dimensi

In [7]:
# ============= CONTOH PENGGUNAAN =============
# Uncomment salah satu dari contoh di bawah untuk testing

# Testing dengan test.jpeg jika ada
# results = perform_inference('rottenvsfresh_classifier/test.jpeg', measure_dimensions=False)

# Testing dengan image lain
# results = perform_inference('fruit_dimension/sample/1.png', measure_dimensions=True, visualize_dims=True)

# Untuk menggunakan, uncomment baris di atas dan jalankan cell ini

In [8]:
# ============= HELPER FUNCTION UNTUK BATCH INFERENCE =============
def batch_inference(image_dir, pattern='*.jpg', measure_dimensions=False):
    """
    Melakukan inference pada multiple images dalam satu direktori
    
    Parameters:
    -----------
    image_dir : str
        Path ke direktori yang berisi images
    pattern : str
        Pattern file (default: *.jpg)
    measure_dimensions : bool
        Apakah akan mengukur dimensi
    
    Returns:
    --------
    list : List hasil inference untuk semua images
    """
    import glob
    
    if not os.path.exists(image_dir):
        print(f"✗ Directory not found: {image_dir}")
        return []
    
    image_paths = glob.glob(os.path.join(image_dir, pattern))
    
    if not image_paths:
        print(f"✗ No images found in {image_dir} with pattern {pattern}")
        return []
    
    all_results = []
    for i, img_path in enumerate(image_paths, 1):
        print(f"\nProcessing [{i}/{len(image_paths)}]")
        result = perform_inference(img_path, measure_dimensions=measure_dimensions)
        all_results.append(result)
    
    return all_results

# Contoh batch inference:
# results = batch_inference('path/to/image/directory', pattern='*.jpg')

In [9]:
# ============= SUMMARY & STATISTICS =============
def print_summary(inference_results):
    """
    Menampilkan summary dari hasil inference
    """
    if not isinstance(inference_results, list):
        inference_results = [inference_results]
    
    print("\n" + "="*60)
    print("INFERENCE SUMMARY")
    print("="*60)
    print(f"Total images processed: {len(inference_results)}")
    
    if inference_results:
        # Average inference times
        fruit_times = []
        rotten_times = []
        dim_times = []
        
        for result in inference_results:
            if result is None:
                continue
            if 'fruit_classification' in result and result['fruit_classification']:
                fruit_times.append(result['fruit_classification'].get('inference_time', 0))
            if 'rotten_fresh_classification' in result and result['rotten_fresh_classification']:
                rotten_times.append(result['rotten_fresh_classification'].get('inference_time', 0))
            if 'dimensions' in result and result['dimensions']:
                dim_times.append(result['dimensions'].get('inference_time', 0))
        
        if fruit_times:
            print(f"\nFruit Classification - Avg Time: {np.mean(fruit_times):.4f}s")
        if rotten_times:
            print(f"Rotten vs Fresh - Avg Time: {np.mean(rotten_times):.4f}s")
        if dim_times:
            print(f"Dimension Measurement - Avg Time: {np.mean(dim_times):.4f}s")
        
        print("\n" + "="*60)

# Contoh:
# print_summary(results)